# Agent Behavior Evaluation with Strands Evals

This notebook evaluates the **LLMEvaluatorAgent's behavior** — not the LLM output quality,
but whether the agent itself makes the right decisions.

| What we evaluate | Framework | Focus |
|-----------------|-----------|-------|
| LLM output quality | MLflow + DeepEval (see `llm_evaluator_agent.ipynb`) | Is the answer correct, fluent, safe? |
| **Agent behavior** | **Strands Evals (this notebook)** | **Did the agent pick the right tools? Right order? Was it helpful?** |

### Evaluators used

| Evaluator | What it measures |
|-----------|------------------|
| `OutputEvaluator` | Is the agent's final response accurate and helpful? |
| `TrajectoryEvaluator` | Did the agent call the right tools in the right order? |
| `ToolSelectionAccuracyEvaluator` | Did the agent pick the correct tools? |
| `HelpfulnessEvaluator` | Was the agent genuinely helpful? (trace-based) |
| `FaithfulnessEvaluator` | Did the agent stay faithful to tool outputs? (trace-based) |

## 1. Setup

In [1]:
import os
import sys; sys.path.insert(0, os.path.abspath(".."))
os.environ["AWS_REGION"] = "ca-central-1"
# os.environ["AGENT_ROLE_ARN"] = "arn:aws:iam::<ACCOUNT_ID>:role/llm-eval-agent-execution"

import nest_asyncio
nest_asyncio.apply()

import logging
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

try:
    from multiprocess.resource_tracker import ResourceTracker
    ResourceTracker.__del__ = lambda self: None
except Exception:
    pass

from src import config
from src.tools import _ensure_aws_env_vars
_ensure_aws_env_vars()

/Users/anuprm/Documents/Code/rbcevals/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Create the Agent

We create the same agent as in `llm_evaluator_agent.py`, but with `callback_handler=None`
to suppress console output during automated evaluation.

In [2]:
from strands import Agent
from strands.models import BedrockModel
from src.tools import (
    load_evaluation_dataset,
    run_bedrock_evaluation,
    run_all_evaluations,
    get_experiment_summary,
)

TOOLS = [load_evaluation_dataset, run_bedrock_evaluation, run_all_evaluations, get_experiment_summary]

def create_agent(**kwargs):
    """Create a fresh agent instance for each test case."""
    model = BedrockModel(
        model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        boto_session=config.get_boto_session(),
    )
    return Agent(
        model=model,
        name="LLMEvaluatorAgent",
        tools=TOOLS,
        system_prompt="You are an LLM Evaluation Agent. You help users evaluate language models using standardized datasets and MLflow tracking.",
        callback_handler=None,
        **kwargs,
    )

print("Agent factory ready")

Agent factory ready


## 3. Define Test Cases

Each test case specifies:
- **input**: What the user asks the agent
- **expected_output**: What we expect in the response
- **expected_trajectory**: Which tools should be called, in order

In [3]:
from strands_evals import Case

test_cases = [
    Case[str, str](
        name="load-dataset",
        input="Load 5 samples from the dataset",
        expected_output="Loaded 5 samples",
        expected_trajectory=["load_evaluation_dataset"],
    ),
    Case[str, str](
        name="eval-single-model",
        input="Load 3 samples and evaluate claude-sonnet-4-5",
        expected_output="Evaluation complete",
        expected_trajectory=["load_evaluation_dataset", "run_bedrock_evaluation"],
    ),
    Case[str, str](
        name="get-summary",
        input="Show me a summary of the experiment results",
        expected_output="Experiment summary",
        expected_trajectory=["get_experiment_summary"],
    ),
]

print(f"{len(test_cases)} test cases defined")

3 test cases defined


## 4. Define Task Functions

Strands Evals calls these functions for each test case. They run the agent and return
the output + trajectory (or traces) for evaluation.

In [4]:
from strands_evals.extractors import tools_use_extractor
from strands_evals.telemetry import StrandsEvalsTelemetry
from strands_evals.mappers import StrandsInMemorySessionMapper

telemetry = StrandsEvalsTelemetry().setup_in_memory_exporter()

def trajectory_task(case: Case) -> dict:
    """Run agent, return output + tool trajectory."""
    agent = create_agent()
    response = agent(case.input)
    trajectory = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
    return {"output": str(response), "trajectory": trajectory}

def trace_task(case: Case) -> dict:
    """Run agent with telemetry, return output + session traces."""
    telemetry.in_memory_exporter.clear()
    agent = create_agent(
        trace_attributes={
            "gen_ai.conversation.id": case.session_id,
            "session.id": case.session_id,
        },
    )
    response = agent(case.input)
    spans = telemetry.in_memory_exporter.get_finished_spans()
    session = StrandsInMemorySessionMapper().map_to_session(spans, session_id=case.session_id)
    return {"output": str(response), "trajectory": session}

print("Task functions ready")

Task functions ready


## 5. Configure Evaluators

All evaluators use **Bedrock Claude Sonnet 4.5** as the judge model — no OpenAI needed.

In [5]:
from strands_evals.evaluators import (
    OutputEvaluator,
    TrajectoryEvaluator,
    ToolSelectionAccuracyEvaluator,
    HelpfulnessEvaluator,
    FaithfulnessEvaluator,
)

judge_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    boto_session=config.get_boto_session(),
)

output_evaluator = OutputEvaluator(
    model=judge_model,
    rubric="""
    Evaluate the agent's response:
    1. Did it correctly perform the requested action?
    2. Did it report results clearly?
    3. Was the response well-structured?
    Score 1.0 if excellent, 0.5 if partial, 0.0 if wrong.
    """,
    include_inputs=True,
)

# Trajectory evaluator needs tool descriptions
sample_agent = create_agent()
tool_descriptions = tools_use_extractor.extract_tools_description(sample_agent, is_short=True)

trajectory_evaluator = TrajectoryEvaluator(
    model=judge_model,
    rubric="""
    Evaluate tool usage:
    1. Were the correct tools selected?
    2. Were they called in the right order?
    3. Were unnecessary tools avoided?
    Score 1.0 if optimal, 0.5 if suboptimal, 0.0 if wrong.
    """,
    include_inputs=True,
)
trajectory_evaluator.update_trajectory_description(tool_descriptions)

tool_selection_evaluator = ToolSelectionAccuracyEvaluator(model=judge_model)
helpfulness_evaluator = HelpfulnessEvaluator(model=judge_model)
faithfulness_evaluator = FaithfulnessEvaluator(model=judge_model)

print("5 evaluators configured, all using Bedrock as judge")

5 evaluators configured, all using Bedrock as judge


## 6. Run: Output + Trajectory Evaluation

This pass runs the agent once per test case and evaluates:
- **Output quality** — was the response helpful and accurate?
- **Trajectory** — did it call the right tools in the right order?
- **Tool selection** — did it pick the correct tools?

In [6]:
from strands_evals import Experiment

exp = Experiment[str, str](
    cases=test_cases,
    evaluators=[output_evaluator, trajectory_evaluator, tool_selection_evaluator],
)
reports = exp.run_evaluations(trajectory_task)

for report in reports:
    print(f"\n📊 {report.evaluator_name}")
    print(f"   Score: {report.overall_score:.2f}  |  Pass: {sum(report.test_passes)}/{len(report.test_passes)}")
    for i, (score, passed, reason) in enumerate(zip(report.scores, report.test_passes, report.reasons)):
        status = "✅" if passed else "❌"
        print(f"   {status} {test_cases[i].name}: {score:.2f} — {reason[:120]}")

2026/04/22 10:53:11 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/04/22 10:53:11 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?]

KeyboardInterrupt: 

## 7. Run: Trace-based Evaluation

This pass captures OpenTelemetry traces and evaluates:
- **Helpfulness** — was the agent genuinely helpful?
- **Faithfulness** — did the agent stay faithful to tool outputs?

In [ ]:
trace_exp = Experiment[str, str](
    cases=test_cases,
    evaluators=[helpfulness_evaluator, faithfulness_evaluator],
)
trace_reports = trace_exp.run_evaluations(trace_task)

for report in trace_reports:
    print(f"\n📊 {report.evaluator_name}")
    print(f"   Score: {report.overall_score:.2f}  |  Pass: {sum(report.test_passes)}/{len(report.test_passes)}")
    for i, (score, passed, reason) in enumerate(zip(report.scores, report.test_passes, report.reasons)):
        status = "✅" if passed else "❌"
        print(f"   {status} {test_cases[i].name}: {score:.2f} — {reason[:120]}")

## 8. Log Results to MLflow

Log all evaluation results to the same MLflow App for a unified view.

In [ ]:
import json
import mlflow

mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
mlflow.set_experiment("agent-behavior-evaluation")

all_reports = reports + trace_reports
all_results = {}

with mlflow.start_run(run_name="agent-behavior-eval"):
    mlflow.log_param("num_test_cases", len(test_cases))
    mlflow.log_param("evaluators", [r.evaluator_name for r in all_reports])

    for report in all_reports:
        name = report.evaluator_name
        mlflow.log_metric(f"{name}/mean_score", report.overall_score)
        mlflow.log_metric(f"{name}/pass_rate", sum(report.test_passes) / len(report.test_passes))
        all_results[name] = {
            test_cases[i].name: {"score": report.scores[i], "passed": report.test_passes[i], "reason": report.reasons[i]}
            for i in range(len(test_cases))
        }

    with open("/tmp/agent_behavior_results.json", "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    mlflow.log_artifact("/tmp/agent_behavior_results.json")

print("✅ Results logged to MLflow experiment: agent-behavior-evaluation")

## 9. Generate EMRM Evaluation Report

Generate a .docx report from the EMRM template with the agent behavior results.

In [ ]:
from src.report_generator import generate_agent_behavior_report

filepath = generate_agent_behavior_report(all_results, len(test_cases))
print(f"📄 Report saved: {filepath}")

## Summary

This notebook demonstrated:

1. **Strands Evals test cases** — define inputs, expected outputs, and expected tool trajectories
2. **Output evaluation** — LLM-as-Judge assesses the agent's final response quality
3. **Trajectory evaluation** — verifies the agent called the right tools in the right order
4. **Tool selection accuracy** — checks if the agent picked the correct tools
5. **Trace-based evaluation** — helpfulness and faithfulness via OpenTelemetry traces
6. **MLflow integration** — all results logged for unified tracking

Combined with the LLM output evaluation in `llm_evaluator_agent.ipynb`, you get a complete
picture of both **what the model produces** and **how the agent behaves**.